# PABPC1–RNA structural preflight (incomplete research tool)

**This is an audit notebook, not the requested completed minimization and mutation pipeline.** It does not generate a relaxed PDB or model additional nucleotides. The earlier soft-repair method must not be treated as stereochemically validated. It cannot establish biological correctness of a pose.

Inputs: `revised_template.pdb` (or `revised_template(1).pdb`), `Muller_benchmark.xlsx`, and `Muller_Position_Top_Two_Chemical_Modification_Library.xlsx`. The workbooks are optional for the structural audit. Outputs: JSON diagnostic, planning-only length CSV, and a ZIP. **The earlier `temp_complex.pdb` is a different 30-nt structure; do not substitute it for the requested input.**

Sequence convention in this notebook: take the 16 observed UTR bases ending at the last non-A and preserve any native 3′ terminal A run, then append A15. Such constructs can exceed 31 nt and are not currently modeled. The 5′ terminus is intended to be hydroxylated; terminal chemistry requires further structural and force-field checks.

## 1. Install a pinned environment
This cell may take several minutes and requires network access. Restart the runtime if Colab reports a package conflict.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openmm==8.6.1', 'numpy==2.4.6', 'scipy==1.17.1', 'biopython==1.88', 'openpyxl==3.1.5'], check=True)


## 2. Install the bundled audit code
This is embedded inside the notebook; no GitHub download is needed.

In [ ]:
from pathlib import Path
PROJECT = Path('/content/PABPC1_RNA_preflight')
(PROJECT/'tests').mkdir(parents=True, exist_ok=True)
(PROJECT/'reports').mkdir(exist_ok=True)
(PROJECT/'inputs').mkdir(exist_ok=True)
(PROJECT/'preflight.py').write_text('"""Conservative structural audit and input checks; not a minimization pipeline."""\nimport csv,json,math,hashlib,re\nfrom collections import Counter\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.spatial import cKDTree\nfrom Bio.PDB import PDBParser\nfrom Bio.SeqUtils import seq1\n\nK_RESTRAINT=4184.0 # kJ mol^-1 nm^-2 for 0.5*k*r^2, 10 kcal mol^-1 A^-2\nHEADERS=[\'variant_id\',\'group\',\'rna_sequence\',\'modifications_json\',\'replicate_id\',\'notes\']\nRNA={\'A\',\'C\',\'G\',\'U\'}\nSUGAR={"O5\'","C5\'","C4\'","O4\'","C3\'","O3\'","C2\'","O2\'","C1\'"}\nBASE={\'A\':{\'N9\',\'C8\',\'N7\',\'C5\',\'C6\',\'N6\',\'N1\',\'C2\',\'N3\',\'C4\'}, \'G\':{\'N9\',\'C8\',\'N7\',\'C5\',\'C6\',\'O6\',\'N1\',\'C2\',\'N2\',\'N3\',\'C4\'},\'C\':{\'N1\',\'C2\',\'O2\',\'N3\',\'C4\',\'N4\',\'C5\',\'C6\'},\'U\':{\'N1\',\'C2\',\'O2\',\'N3\',\'C4\',\'O4\',\'C5\',\'C6\'}}\n\ndef sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\ndef validate_variants(path,expected_counts={\'benchmark\':34,\'control\':27}):\n with open(path,newline=\'\',encoding=\'utf-8-sig\') as f:\n  rd=csv.DictReader(f)\n  if rd.fieldnames!=HEADERS: raise ValueError(\'CSV headers must exactly equal \'+\',\'.join(HEADERS))\n  rows=list(rd)\n seen=set()\n for i,r in enumerate(rows,2):\n  if None in r or any(v is None for v in r.values()): raise ValueError(f\'Row {i}: incorrect number of fields\')\n  v=r[\'variant_id\'].strip(); group=r[\'group\']; seq=r[\'rna_sequence\'].strip().upper()\n  if not re.fullmatch(r\'[A-Za-z0-9_.-]+\',v) or v in (\'.\',\'..\') or v in seen: raise ValueError(f\'Row {i}: invalid or duplicate variant_id\')\n  seen.add(v)\n  if group not in (\'benchmark\',\'control\') or not r[\'replicate_id\'].strip(): raise ValueError(f\'Row {i}: invalid group or replicate ID\')\n  if len(seq)!=31 or set(seq)-RNA: raise ValueError(f\'Row {i}: sequence must contain exactly 31 A/C/G/U characters; variable-length structures require a separate validated builder\')\n  try: mods=json.loads(r[\'modifications_json\'])\n  except json.JSONDecodeError as e: raise ValueError(f\'Row {i}: malformed modifications JSON: {e}\') from e\n  if not isinstance(mods,list): raise ValueError(f\'Row {i}: modifications must be an array\')\n  used=set()\n  for m in mods:\n   if not isinstance(m,dict) or set(m)!={\'position\',\'modification\',\'parent_base\'}: raise ValueError(f\'Row {i}: each modification requires position, modification, parent_base\')\n   pos=m[\'position\']; parent=m[\'parent_base\']; mod=m[\'modification\']\n   if type(pos) is not int or not 1<=pos<=31 or pos in used or parent not in RNA or parent!=seq[pos-1] or not isinstance(mod,str) or not mod.strip(): raise ValueError(f\'Row {i}: invalid modification at {pos}\')\n   if mod==\'m6A\' and parent!=\'A\': raise ValueError(f\'Row {i}: m6A requires A\')\n   used.add(pos)\n  if (seq[16:]!=\'A\'*15 or any(p>=17 for p in used)) and \'[allow_A15_changes]\' not in r[\'notes\']: raise ValueError(f\'Row {i}: A15 changes require [allow_A15_changes] in notes\')\n  r[\'rna_sequence\']=seq; r[\'modification_status\']=\'requires_parameterization\' if mods else \'unmodified\'; r[\'parsed_modifications\']=mods\n if expected_counts is not None and Counter(r[\'group\'] for r in rows)!=Counter(expected_counts): raise ValueError(f\'Unexpected group counts: {dict(Counter(r["group"] for r in rows))}; expected {expected_counts}\')\n return rows\n\ndef audit_pdb(path,uniprot_sequence=None):\n path=Path(path); lines=path.read_text(errors=\'replace\').splitlines(); cryst=next((x for x in lines if x.startswith(\'CRYST1\')),None)\n box=None\n if cryst:\n  try: box=[float(cryst[6+i:15+i]) for i in (0,9,18)]\n  except ValueError: box=None\n placeholder=bool(box and min(box)<=2.0)\n models=sum(x.startswith(\'MODEL \') for x in lines)\n atomlines=[x for x in lines if x.startswith((\'ATOM  \',\'HETATM\'))]\n keys=[(x[21:22],x[22:27],x[12:16],x[16:17],x[17:20]) for x in atomlines]\n duplicates=[str(k) for k,n in Counter(keys).items() if n>1]\n altloc=sum(bool(x[16:17].strip()) for x in atomlines)\n parser=PDBParser(QUIET=True,PERMISSIVE=False)\n structure=parser.get_structure(\'input\',str(path)); model=next(structure.get_models())\n chains={c.id:list(c.get_residues()) for c in model}; protein=chains.get(\'A\',[]); rna=chains.get(\'E\',[])\n def one(r): return seq1(r.resname,custom_map={\'UNK\':\'X\'},undef_code=\'X\')\n protseq=\'\'.join(one(r) for r in protein); rnaseq=\'\'.join(r.resname.strip() if r.resname.strip() in RNA else \'?\' for r in rna)\n ids=[r.id[1] for r in rna]\n missing=[]\n for j,r in enumerate(rna):\n  base=r.resname.strip()\n  if base not in BASE: missing.append({\'residue\':r.id[1],\'error\':\'unsupported residue \'+base}); continue\n  need=SUGAR|BASE[base]\n  if j>0: need=need|{\'P\',\'OP1\',\'OP2\'}\n  m=sorted(need-{a.name for a in r})\n  if m: missing.append({\'residue\':r.id[1],\'missing_heavy_atoms\':m})\n for r in protein:\n  m=sorted({\'N\',\'CA\',\'C\',\'O\'}-{a.name for a in r})\n  if m: missing.append({\'chain\':\'A\',\'residue\':r.id[1],\'missing_backbone_atoms\':m})\n def heavy(rs): return [a for r in rs for a in r if a.element not in (\'H\',\'D\') and a.element]\n pa,ra=heavy(protein),heavy(rna); contacts={\'pairs_lt_2A\':None,\'minimum_A\':None,\'closest_pair\':None,\'criterion\':\'all protein–RNA heavy-atom pairs < 2.0 Å; no interchain covalent bonds assumed\'}\n if pa and ra:\n  px=np.array([a.coord for a in pa]); rx=np.array([a.coord for a in ra]); tree=cKDTree(px); close=tree.query_ball_point(rx,2.0)\n  contacts[\'pairs_lt_2A\']=sum(float(np.linalg.norm(rx[j]-px[i]))<2.0 for j,ix in enumerate(close) for i in ix)\n  dist,idx=tree.query(rx); j=int(np.argmin(dist)); a,b=pa[int(idx[j])],ra[j]\n  def atomid(x): return {\'chain\':x.parent.parent.id,\'residue\':x.parent.id[1],\'resname\':x.parent.resname,\'atom\':x.name}\n  contacts[\'minimum_A\']=float(dist[j]);contacts[\'closest_pair\']=[atomid(a),atomid(b)]\n bonds=[]\n for left,right in zip(rna,rna[1:]):\n  d=float(np.linalg.norm(left["O3\'"].coord-right[\'P\'].coord)) if "O3\'" in left and \'P\' in right else None\n  bonds.append({\'from_residue\':left.id[1],\'to_residue\':right.id[1],\'O3_P_A\':d,\'within_1_3_to_2_0_A\':d is not None and 1.3<=d<=2.0})\n peptide=[]\n for left,right in zip(protein,protein[1:]):\n  d=float(np.linalg.norm(left[\'C\'].coord-right[\'N\'].coord)) if \'C\' in left and \'N\' in right else None\n  if d is None or not 1.1<=d<=1.8: peptide.append({\'from\':left.id[1],\'to\':right.id[1],\'C_N_A\':d})\n five_prime_atoms=sorted(a.name for a in rna[0]) if rna else []\n five_prime_phosphate=any(n in five_prime_atoms for n in (\'P\',\'OP1\',\'OP2\',\'OP3\'))\n errors=[]\n if five_prime_phosphate: errors.append(\'5-prime terminal phosphate atoms conflict with the requested 5-prime hydroxyl; explicit chemical conversion is required\')\n if placeholder: errors.append(\'CRYST1 placeholder box must be removed or overridden before periodic simulation\')\n if models>1: errors.append(\'Multiple MODEL records\')\n if duplicates or altloc: errors.append(\'Duplicate atoms or alternate locations require explicit review\')\n if len(protein)!=360 or [r.id[1] for r in protein]!=list(range(1,361)): errors.append(\'Chain A is not the expected residues 1–360\')\n if ids!=list(range(37,68)) or rnaseq!=\'A\'*31: errors.append(\'Chain E is not expected 31 adenines E37–E67\')\n if uniprot_sequence is not None and protseq!=uniprot_sequence[10:370]: errors.append(\'Chain A sequence does not exactly match P11940 residues 11–370\')\n if missing: errors.append(\'Missing expected heavy atoms or unsupported residues\')\n if any(not b[\'within_1_3_to_2_0_A\'] for b in bonds): errors.append(\'RNA linkage outlier or missing bond atoms\')\n if peptide: errors.append(\'Protein peptide connectivity outlier\')\n if contacts[\'minimum_A\'] is not None and contacts[\'minimum_A\']<2: errors.append(\'Protein–RNA contacts < 2 Å require structural review\')\n return {\'input_file\':path.name,\'sha256\':sha256(path),\'model_records\':models,\'chain_residue_counts\':{k:len(v) for k,v in chains.items()},\'protein_sequence\':protseq,\'uniprot_mapping_11_370_match\':None if uniprot_sequence is None else protseq==uniprot_sequence[10:370], \'rna_sequence\':rnaseq,\'rna_mapping\':[{\'position\':j+1,\'chain\':\'E\',\'residue\':r.id[1],\'base\':r.resname} for j,r in enumerate(rna)],\'rna_termini\':{\'five_prime_residue\':rna[0].id[1] if rna else None,\'five_prime_atoms\':five_prime_atoms,\'five_prime_phosphate_present\':five_prime_phosphate,\'intended_five_prime_chemistry\':\'5-prime hydroxyl\',\'intended_three_prime_chemistry\':\'3-prime hydroxyl; verify\'},\'cryst1\':{\'line\':cryst,\'lengths_A\':box,\'placeholder_detected\':placeholder},\'duplicate_atom_records\':duplicates,\'alternate_location_atom_records\':altloc,\'missing_atoms\':missing,\'rna_linkages\':bonds,\'peptide_linkage_outliers\':peptide,\'contacts\':contacts,\'preflight_errors\':errors,\'limitations\':[\'Side-chain completeness, absolute stereochemistry and force-field matching are not fully validated by this audit.\',\'A 5-prime hydroxyl and 3-prime hydroxyl are intended but terminal chemistry needs explicit atom/bond and template verification.\',\'No relaxation or repair was performed.\']}\n\ndef normalized_length_analysis(utr50,utr15):\n if not isinstance(utr50,str) or not isinstance(utr15,str): raise ValueError(\'missing sequence\')\n s=utr50.upper().strip(); short=utr15.upper().strip()\n if len(s)!=50 or len(short)!=15 or s[-15:]!=short or set(s)-RNA: raise ValueError(\'UTR50 does not agree with UTR15\')\n run=len(s)-len(s.rstrip(\'A\')); end=50-run\n if end<16: raise ValueError(\'fewer than 16 observed bases through the last non-A; cannot derive construct\')\n return {\'native_terminal_A_count\':run,\'normalized_UTR16\':s[end-16:end],\'candidate_sequence\':s[end-16:]+\'A\'*15,\'candidate_length\':31+run,\'fixed_31_supported\':run==0}\n')
(PROJECT/'workbook_report.py').write_text('"""Analyze actual workbooks; planning report only, no structures or accepted variants CSV."""\nimport sys,csv,json\nfrom collections import Counter\nfrom pathlib import Path\nfrom openpyxl import load_workbook\nfrom preflight import normalized_length_analysis\n\ndef read(path,sheet,idcol,longcol,shortcol,group):\n ws=load_workbook(path,read_only=True,data_only=True)[sheet]; it=ws.values; headers=next(it)\n rows=[]\n for values in it:\n  r=dict(zip(headers,values))\n  if not r[idcol]: continue\n  info=normalized_length_analysis(r[longcol],r[shortcol])\n  rows.append({\'group\':group,\'source_id\':r[idcol],**info,\'status\':\'planning_only_template_extension_required\' if not info[\'fixed_31_supported\'] else \'planning_only_not_structurally_validated\'})\n return rows\n\ndef write_candidates(rows,path):\n headers=[\'variant_id\',\'group\',\'rna_sequence\',\'modifications_json\',\'replicate_id\',\'notes\']\n seen=set()\n with open(path,\'w\',newline=\'\',encoding=\'utf-8\') as f:\n  writer=csv.DictWriter(f,fieldnames=headers);writer.writeheader()\n  for r in rows:\n   identifier=(\'BM_\' if r[\'group\']==\'benchmark\' else \'CTRL_\')+str(r[\'source_id\'])\n   if identifier in seen:raise ValueError(\'Duplicate candidate identifier: \'+identifier)\n   seen.add(identifier);n=int(r[\'native_terminal_A_count\']);seq=r[\'candidate_sequence\']\n   if len(seq)!=31+n or seq[-(15+n):]!=\'A\'*(15+n):raise ValueError(\'Inconsistent candidate sequence: \'+identifier)\n   condition=\'template extension required\' if n else \'31-nt construct; structural preparation still required\'\n   writer.writerow({\'variant_id\':identifier,\'group\':r[\'group\'],\'rna_sequence\':seq,\'modifications_json\':\'[]\',\'replicate_id\':\'1\',\'notes\':f"Planning only; source {r[\'source_id\']}; 16 nt to last non-A + {n} native A + A15; {condition}"})\n\ndef main(b,c,out,candidate_path=None):\n rows=read(b,\'Selected panel\',\'RNA_ID\',\'UTR50\',\'UTR15\',\'benchmark\')+read(c,\'Unmodified controls\',\'RNA ID\',\'50-nt native UTR\',\'15-nt native UTR\',\'control\')\n if Counter(r[\'group\'] for r in rows)!=Counter({\'benchmark\':34,\'control\':27}): raise ValueError(\'Workbook counts differ from expected 34 and 27\')\n with open(out,\'w\',newline=\'\') as f:\n  w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\n if candidate_path:write_candidates(rows,candidate_path)\n print(json.dumps({\'counts\':dict(Counter(r[\'group\'] for r in rows)),\'length_counts\':dict(sorted(Counter(r[\'candidate_length\'] for r in rows).items())),\'longer_than_31\':sum(not r[\'fixed_31_supported\'] for r in rows),\'report\':str(out)},indent=2))\nif __name__==\'__main__\': main(*sys.argv[1:])\n')
(PROJECT/'tests/test_preflight.py').write_text('import unittest, tempfile, csv, json\nfrom pathlib import Path\nimport sys\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom preflight import validate_variants, audit_pdb, K_RESTRAINT, normalized_length_analysis\n\nclass Validate(unittest.TestCase):\n def csvfile(self, rows):\n  d=tempfile.TemporaryDirectory(); self.addCleanup(d.cleanup); p=Path(d.name)/\'v.csv\'\n  with p.open(\'w\',newline=\'\') as f:\n   w=csv.writer(f); w.writerow([\'variant_id\',\'group\',\'rna_sequence\',\'modifications_json\',\'replicate_id\',\'notes\']); w.writerows(rows)\n  return p\n def row(self,**kw):\n  d={\'variant_id\':\'b1\',\'group\':\'benchmark\',\'rna_sequence\':\'A\'*31,\'modifications_json\':\'[]\',\'replicate_id\':\'1\',\'notes\':\'\'}; d.update(kw); return list(d.values())\n def test_valid(self): self.assertEqual(len(validate_variants(self.csvfile([self.row()]),expected_counts=None)),1)\n def test_duplicate_id(self):\n  with self.assertRaises(ValueError): validate_variants(self.csvfile([self.row(),self.row()]),None)\n def test_bad_base(self):\n  with self.assertRaises(ValueError): validate_variants(self.csvfile([self.row(rna_sequence=\'A\'*30+\'T\')]),None)\n def test_short(self):\n  with self.assertRaises(ValueError): validate_variants(self.csvfile([self.row(rna_sequence=\'A\'*30)]),None)\n def test_tail(self):\n  with self.assertRaises(ValueError): validate_variants(self.csvfile([self.row(rna_sequence=\'A\'*30+\'G\')]),None)\n def test_tail_authorized(self): self.assertEqual(len(validate_variants(self.csvfile([self.row(rna_sequence=\'A\'*30+\'G\',notes=\'[allow_A15_changes]\')]),None)),1)\n def test_bad_parent(self):\n  with self.assertRaises(ValueError): validate_variants(self.csvfile([self.row(modifications_json=\'[{"position": 2, "modification": "m6A", "parent_base": "C"}]\')]),None)\n def test_duplicate_sites(self):\n  with self.assertRaises(ValueError): validate_variants(self.csvfile([self.row(modifications_json=\'[{"position": 2,"modification":"m6A","parent_base":"A"},{"position": 2,"modification":"2OMe","parent_base":"A"}]\')]),None)\n def test_modification_is_blocked(self):\n  rows=validate_variants(self.csvfile([self.row(modifications_json=\'[{"position": 2,"modification":"m6A","parent_base":"A"}]\')]),None)\n  self.assertEqual(rows[0][\'modification_status\'],\'requires_parameterization\')\n def test_native_terminal_as_preserved(self):\n  a=normalized_length_analysis(\'G\'*47+\'AAA\',\'G\'*12+\'AAA\')\n  self.assertEqual(a[\'candidate_length\'],34)\n  self.assertEqual(a[\'candidate_sequence\'],\'G\'*16+\'A\'*18)\n  self.assertFalse(a[\'fixed_31_supported\'])\n def test_utr_mismatch(self):\n  with self.assertRaises(ValueError): normalized_length_analysis(\'G\'*50,\'C\'*15)\n def test_units(self): self.assertEqual(K_RESTRAINT,4184.0)\n def test_invalid_box_and_contact(self):\n  text=(\'CRYST1    1.000    1.000    1.000  90.00  90.00  90.00 P 1           1\\n\'\n  \'ATOM      1   CA ALA A   1       0.000   0.000   0.000  1.00  0.00           C\\n\'\n  "ATOM      2  C1\'   A E  37       1.000   0.000   0.000  1.00  0.00           C\\nEND\\n")\n  with tempfile.TemporaryDirectory() as d:\n   p=Path(d)/\'tiny.pdb\'; p.write_text(text); a=audit_pdb(p,None)\n   self.assertTrue(a[\'cryst1\'][\'placeholder_detected\']); self.assertEqual(a[\'contacts\'][\'pairs_lt_2A\'],1)\n def test_5prime_phosphate_and_placeholder_box_are_flagged(self):\n  text=("CRYST1    1.000    1.000    1.000  90.00  90.00  90.00 P 1           1\\n"\n        "ATOM      1    P   A E  37       1.000   0.000   0.000  1.00  0.00           P\\n"\n        "ATOM      2  OP1   A E  37       1.000   1.000   0.000  1.00  0.00           O\\n"\n        "ATOM      3  OP2   A E  37       1.000  -1.000   0.000  1.00  0.00           O\\nEND\\n")\n  with tempfile.TemporaryDirectory() as d:\n   p=Path(d)/\'tiny.pdb\'; p.write_text(text); a=audit_pdb(p,None)\n   self.assertTrue(a[\'rna_termini\'][\'five_prime_phosphate_present\'])\n   self.assertTrue(any(\'5-prime\' in x for x in a[\'preflight_errors\']))\n   self.assertTrue(any(\'CRYST1\' in x for x in a[\'preflight_errors\']))\n def test_counts(self):\n  with self.assertRaises(ValueError): validate_variants(self.csvfile([self.row()]))\nif __name__==\'__main__\': unittest.main()\n')
(PROJECT/'tests/test_physics.py').write_text("import unittest\nimport numpy as np\nimport openmm as mm\nfrom openmm import unit, app\n\nclass OpenMMChecks(unittest.TestCase):\n def test_forcefields_load(self):\n  app.ForceField('amber19/protein.ff19SB.xml','amber14/RNA.OL3.xml','amber19/tip3p.xml')\n def test_actual_minimization_and_restraint(self):\n  system=mm.System(); system.addParticle(12.0)\n  f=mm.CustomExternalForce('0.5*k*(x-x0)^2'); f.addGlobalParameter('k',4184.0); f.addPerParticleParameter('x0'); f.addParticle(0,[0.0]); system.addForce(f)\n  integrator=mm.VerletIntegrator(0.001); ctx=mm.Context(system,integrator,mm.Platform.getPlatformByName('Reference')); ctx.setPositions([[0.1,0.,0.]])\n  before=ctx.getState(getEnergy=True,getForces=True); e=before.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)\n  self.assertAlmostEqual(e,20.92,places=6)\n  mm.LocalEnergyMinimizer.minimize(ctx,1e-6,100)\n  state=ctx.getState(getEnergy=True,getPositions=True); after=state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)\n  self.assertTrue(np.isfinite(after));self.assertLess(after,1e-6)\nif __name__=='__main__': unittest.main()\n")
(PROJECT/'tests/test_workbook_report.py').write_text("import unittest, tempfile, csv, sys\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom workbook_report import write_candidates\nclass CandidateTests(unittest.TestCase):\n def test_full_schema_and_unique_group_ids(self):\n  rows=[{'group':'benchmark','source_id':'RNA_1','candidate_sequence':'G'*16+'A'*15,'native_terminal_A_count':0,'candidate_length':31},\n        {'group':'control','source_id':'RNA_1','candidate_sequence':'G'*16+'A'*18,'native_terminal_A_count':3,'candidate_length':34}]\n  with tempfile.TemporaryDirectory() as d:\n   p=Path(d)/'candidates.csv';write_candidates(rows,p)\n   with p.open(newline='') as f:\n    r=csv.DictReader(f);self.assertEqual(r.fieldnames,['variant_id','group','rna_sequence','modifications_json','replicate_id','notes']);data=list(r)\n   self.assertNotEqual(data[0]['variant_id'],data[1]['variant_id']);self.assertEqual(data[1]['rna_sequence'],'G'*16+'A'*18);self.assertIn('extension required',data[1]['notes'])\nif __name__=='__main__':unittest.main()\n")
import sys
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
print('Project folder:', PROJECT)


## 3. Run automated component tests
The OpenMM minimization is a one-particle mathematical unit test, not a protein–RNA validation.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(PROJECT/'tests'), '-v'], check=True)


## 4. Upload the requested inputs
Upload the PDB and, optionally, both workbooks. Re-running saves the uploaded file with its original filename in `inputs/`.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    if Path(name).name != name: raise ValueError('Invalid uploaded filename')
    (PROJECT/'inputs'/name).write_bytes(data)
print('Available input files:', [p.name for p in (PROJECT/'inputs').iterdir()])


## 5. Audit the exact reference PDB
Fetch the UniProt canonical sequence, inspect chain mapping, selected atom and bond checks, interchain contacts, and the box. The original file is untouched. A positive result is not proof of a correct structure.

In [ ]:
import urllib.request, json
from preflight import audit_pdb
candidates = [PROJECT/'inputs'/name for name in ('revised_template.pdb', 'revised_template(1).pdb') if (PROJECT/'inputs'/name).exists()]
if len(candidates) != 1:
    raise ValueError('Upload exactly one file named revised_template.pdb or revised_template(1).pdb. Earlier structures are not substituted.')
pdb = candidates[0]
with urllib.request.urlopen('https://rest.uniprot.org/uniprotkb/P11940.fasta', timeout=30) as response:
    fasta = response.read().decode()
(PROJECT/'reports'/'P11940.fasta').write_text(fasta)
uniprot = ''.join(fasta.splitlines()[1:])
try:
    report = audit_pdb(pdb, uniprot)
except Exception as e:
    (PROJECT/'reports'/'initial_audit_error.json').write_text(json.dumps({'error':repr(e)}, indent=2))
    raise
(PROJECT/'reports'/'initial_audit.json').write_text(json.dumps(report, indent=2))
for k in ['chain_residue_counts', 'uniprot_mapping_11_370_match', 'contacts', 'rna_termini', 'cryst1', 'preflight_errors', 'limitations']:
    print(k, ':', report[k])
print('Full RNA residue mapping and linkage checks: reports/initial_audit.json')


## 6. Compare the real workbook constructs
This creates a **planning report and candidate CSV**, not structures ready for simulation. A length greater than 31 flags that additional RNA modeling is required.

In [ ]:
import subprocess, sys
b = PROJECT/'inputs'/'Muller_benchmark.xlsx'
c = PROJECT/'inputs'/'Muller_Position_Top_Two_Chemical_Modification_Library.xlsx'
if b.exists() and c.exists():
    subprocess.run([sys.executable, str(PROJECT/'workbook_report.py'), str(b), str(c), str(PROJECT/'reports'/'workbook_length_planning.csv'), str(PROJECT/'reports'/'variants_candidate_not_simulation_ready.csv')], check=True)
else:
    print('Workbook analysis skipped: upload both revised workbooks in cell 4 and rerun this cell.')


## 7. Download the reports
Outputs are in temporary runtime storage; download before the runtime disconnects. The ZIP can contain uploaded input files and sequences, so keep it private.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/PABPC1_RNA_preflight_reports', 'zip', PROJECT)
files.download(zip_path)


## Limitations and next step

The structure audit deliberately stops at detection. It does not establish complete side-chain atom sets, absolute chirality, valid termini, force-field assignment, clash repair, mutation, variable-length RNA extension, or reference minimization. Missing template atoms are not filled in. The raw PDB remains unchanged, including any CRYST1 record; do not load a placeholder unit cell as a simulation box.

A real implementation requires the revised structure, a stereochemistry-preserving clash-resolution strategy validated on it, a verified canonical substitution tool and RNA-extension workflow, plus tests of each stage. Modified nucleotides require separately validated structures and force-field parameters; they cannot be represented by relabeling unmodified bases.

OpenMM XML availability and the harmonic restraint conversion are tested here. ff19SB was developed in an OPC context; TIP3P is an explicit choice and should be evaluated scientifically for the project. OpenMM uses L-BFGS for minimization. 10 kcal mol⁻¹ Å⁻² corresponds to 4184 kJ mol⁻¹ nm⁻² for `E=0.5*k*r²`.

References: [OpenMM force-field guide](https://docs.openmm.org/latest/userguide/application/02_running_sims.html); [ChimeraX swapna source](https://github.com/RBVI/ChimeraX/blob/develop/src/bundles/swap_res/src/cmd.py); [UniProt P11940](https://www.uniprot.org/uniprotkb/P11940/entry).